imports


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Layout
from PIL import Image
import math
import os


In [ ]:
def bresenham_line(x0, y0, x1, y1, shape):
    """Implementacja klasycznego algorytmu Bresenhama dla dyskretnej macierzy."""
    x0, y0, x1, y1 = int(x0), int(y0), int(x1), int(y1)
    h, w = shape
    points = []
    
    dx = abs(x1 - x0)
    dy = abs(y1 - y0)
    sx = 1 if x0 < x1 else -1
    sy = 1 if y0 < y1 else -1
    
    if dx > dy:
        err = dx / 2.0
        while x0 != x1:
            if 0 <= x0 < w and 0 <= y0 < h:
                points.append((y0, x0))
            err -= dy
            if err < 0:
                y0 += sy
                err += dx
            x0 += sx
    else:
        err = dy / 2.0
        while y0 != y1:
            if 0 <= x0 < w and 0 <= y0 < h:
                points.append((y0, x0))
            err -= dx
            if err < 0:
                x0 += sx
                err += dy
            y0 += sy
            
    if 0 <= x0 < w and 0 <= y0 < h:
        points.append((y0, x0))
    return points

def get_positions(alpha, n_detectors, spread, img_shape):
    """Oblicza pozycje emiterów i detektorów jako funkcję kąta alfa."""
    h, w = img_shape
    center = np.array([w // 2, h // 2])
    # Promień musi być duży, by układ był poza obrazem (np. przekątna/2)
    radius = (max(h, w) * 1.42) / 2 + 10 
    
    rad_alpha = math.radians(alpha)
    rad_perp = math.radians(alpha + 90)
    
    # Kierunek główny (E-D) i prostopadły (szerokość rzutu)
    dir_vec = np.array([math.cos(rad_alpha), math.sin(rad_alpha)])
    perp_vec = np.array([math.cos(rad_perp), math.sin(rad_perp)])
    
    # Punkty bazowe
    p0_e = center + radius * dir_vec
    p0_d = center - radius * dir_vec
    
    # Rozmieszczenie n detektorów wzdłuż szerokości 'l' (spread)
    offsets = np.linspace(-spread/2, spread/2, n_detectors)
    
    positions = []
    for off in offsets:
        e = p0_e + off * perp_vec
        d = p0_d + off * perp_vec
        positions.append((e, d))
    return positions

# --- 3. TRANSFORMATY ---
def create_sinogram(image, delta_alpha, n_detectors, spread):
    """Generuje sinogram (Transformata Radona) - Model addytywny."""
    angles = np.arange(0, 180, delta_alpha)
    sinogram = np.zeros((len(angles), n_detectors))
    
    print(f"Obliczanie sinogramu ({len(angles)} rzutów)... Proszę czekać.")
    
    for a_idx, alpha in enumerate(angles):
        positions = get_positions(alpha, n_detectors, spread, image.shape)
        for d_idx, (emitter, detector) in enumerate(positions):
            pixels = bresenham_line(emitter[0], emitter[1], detector[0], detector[1], image.shape)
            if pixels:
                # Normalizacja przez długość linii (średnia absorpcja)
                val = sum(image[p] for p in pixels) / len(pixels)
                sinogram[a_idx, d_idx] = val
    print("Sinogram wygenerowany.")
    return sinogram, angles

def reconstruct_all(sinogram, angles, n_detectors, spread, img_shape):
    """Pełna rekonstrukcja (Backprojection) od zera."""
    output = np.zeros(img_shape)
    
    print(f"Obliczanie pełnej rekonstrukcji... Proszę czekać.")
    for a_idx in range(len(angles)):
        alpha = angles[a_idx]
        positions = get_positions(alpha, n_detectors, spread, img_shape)
        for d_idx, (emitter, detector) in enumerate(positions):
            val = sinogram[a_idx, d_idx]
            pixels = bresenham_line(emitter[0], emitter[1], detector[0], detector[1], img_shape)
            for p in pixels:
                output[p] += val
    
    # Normalizacja końcowa (mapowanie 0-1)
    if np.max(output) > 0:
        output = output / np.max(output)
    print("Rekonstrukcja zakończona.")
    return output

# =========================================================
# --- 4. MIEJSCE NA TWÓJ OBRAZ (DLA OBRAZÓW OK. 400x400) ---
# =========================================================

# WPISZ NAZWĘ SWOJEGO PLIKU TUTAJ (musi być w tym samym folderze lub podaj pełną ścieżkę)
# Przykład: YOUR_IMAGE_PATH = 'moje_pluca.jpg'
YOUR_IMAGE_PATH = 'tomograf-obrazy\\Shepp_logan.jpg' # Pozostaw puste '', aby użyć obrazu testowego

def load_and_preprocess_image(path):
    """Ładuje obraz, konwertuje na skalę szarości i normalizuje."""
    if path and os.path.exists(path):
        try:
            img = Image.open(path).convert('L') # Skala szarości
            img_arr = np.array(img) / 255.0 # Normalizacja 0-1
            print(f"Pomyślnie załadowano obraz: {path} (Rozmiar: {img_arr.shape})")
            return img_arr
        except Exception as e:
            print(f"Błąd podczas ładowania obrazu: {e}. Używam obrazu testowego.")
    else:
        print("Brak pliku lub nie podano ścieżki. Generuję obraz testowy 400x400.")
        
    # Generowanie obrazu testowego 400x400 (fantom)
    size = 400
    fantom = np.zeros((size, size))
    y, x = np.ogrid[:size, :size]
    # Duże koło
    fantom[(x - size//2)**2 + (y - size//2)**2 < (size//3)**2] = 0.5
    # Jasny kwadrat
    fantom[50:150, 50:150] = 1.0
    # Ciemny prostokąt
    fantom[250:350, 200:380] = 0.2
    return fantom

# Ładujemy obraz
fantom = load_and_preprocess_image(YOUR_IMAGE_PATH)
h, w = fantom.shape


# --- 5. PARAMETRY SYMULACJI (DOPASOWANE DO 400x400) ---
# Uwaga: Im większe N i mniejsze D_ALPHA, tym lepsza jakość, ale wolniejsze obliczenia.
N = 400          # Liczba detektorów (n) - ustawione na szerokość obrazu
L = 700          # Rozpiętość układu (l) - musi być > przekątna obrazu (400*1.41 ~ 565)
D_ALPHA = 1.0   # Krok kąta (delta alfa) - dla 400x400 to rozsądny kompromis prędkości.

# Generujemy sinogram
sinogram, angles = create_sinogram(fantom, D_ALPHA, N, L)
# Generujemy pełną rekonstrukcję (żeby suwak cofał się błyskawicznie)
full_reconstruction = reconstruct_all(sinogram, angles, N, L, fantom.shape)

# --- 6. INTERAKTYWNA WIZUALIZACJA (ZOPTYMALIZOWANA DLA 400x400) ---

# Globalne zmienne do keszowania (zoptymalizowana akumulacja)
cached_output = np.zeros(fantom.shape)
last_krok_idx = -1

@interact(krok=IntSlider(min=0, max=len(angles)-1, step=1, value=0, 
                        description='Postęp (Rzuty):', 
                        layout=Layout(width='90%')))
def run_fast_simulation(krok):
    global cached_output, last_krok_idx
    
    # 1. Obliczamy częściową rekonstrukcję (lub używamy pełnej)
    if krok == len(angles)-1:
        # Jeśli suwak jest na końcu, używamy pełnej, policzonej wcześniej rekonstrukcji
        display_reko = full_reconstruction
    elif krok > last_krok_idx:
        # Jeśli suwak poszedł w prawo, doliczamy tylko nowe rzuty (szybkie)
        for a_idx in range(last_krok_idx + 1, krok + 1):
            alpha = angles[a_idx]
            positions = get_positions(alpha, N, L, fantom.shape)
            for d_idx, (emitter, detector) in enumerate(positions):
                val = sinogram[a_idx, d_idx]
                pixels = bresenham_line(emitter[0], emitter[1], detector[0], detector[1], fantom.shape)
                for p in pixels:
                    cached_output[p] += val
        last_krok_idx = krok
        # Normalizacja kopii do wyświetlenia
        display_reko = cached_output / np.max(cached_output) if np.max(cached_output) > 0 else cached_output
    else:
        # Jeśli suwak się cofnął, musimy przeliczyć (wolniejsze, ale rzadsze)
        cached_output = np.zeros(fantom.shape) # Reset cashu
        last_krok_idx = -1
        display_reko = reconstruct_all(sinogram, angles, N, L, fantom.shape) # Uproszczenie dla cofania
    
    # 2. Rysowanie
    fig, ax = plt.subplots(1, 3, figsize=(16, 5))
    
    # Lewy panel: Skaner (rysowany rzadziej dla prędkości)
    ax[0].imshow(fantom, cmap='gray')
    ax[0].set_title(f"Skanowanie (Kąt: {angles[krok]:.1f}°)")
    curr_pos = get_positions(angles[krok], N, L, fantom.shape)
    
    # Rysuj co 15-tą wiązkę, by zachować czytelność na 400x400
    skip = max(1, N // 15)
    for i in range(0, len(curr_pos), skip):
        e, d = curr_pos[i]
        ax[0].plot([e[0], d[0]], [e[1], d[1]], color='red', alpha=0.2, linewidth=0.6)
        ax[0].scatter(e[0], e[1], color='yellow', s=10, zorder=3) # Emiter
        ax[0].scatter(d[0], d[1], color='blue', s=10, zorder=3)   # Detektor
    
    # Ustawiamy widok poza obraz, aby widzieć emiter/detektor
    margin = w // 4
    ax[0].set_xlim(-margin, w + margin)
    ax[0].set_ylim(h + margin, -margin)

    # Środkowy panel: Sinogram
    view_sin = np.zeros_like(sinogram)
    view_sin[:krok+1, :] = sinogram[:krok+1, :]
    ax[1].imshow(view_sin, cmap='magma', aspect='auto')
    ax[1].set_title("Sinogram (Postęp)")
    ax[1].set_ylabel("Kąt (Rzut)")
    ax[1].set_xlabel("Detektor")

    # Prawy panel: Wynik (Wsteczna Projekcja)
    ax[2].imshow(display_reko, cmap='gray')
    ax[2].set_title(f"Rekonstrukcja ({krok+1} rzutów)")
    
    plt.tight_layout()
    plt.show()

Pomyślnie załadowano obraz: tomograf-obrazy\Shepp_logan.jpg (Rozmiar: (1024, 1024))
Obliczanie sinogramu (180 rzutów)... Proszę czekać.
Sinogram wygenerowany.
Obliczanie pełnej rekonstrukcji... Proszę czekać.
Rekonstrukcja zakończona.


interactive(children=(IntSlider(value=0, description='Postęp (Rzuty):', layout=Layout(width='90%'), max=179), …